# NaN Handling

Real-world time series have gaps: failed sensors, market halts, missing ticks.
screamer defines **three NaN policies** that cover every function in the library:

| Policy | Who uses it | Behaviour |
|--------|------------|----------|
| **`ignore`** | Rolling\*, Ew\*, Cum\*, bands, oscillators, filters | NaN skips internal state; one NaN in → one NaN out |
| **`propagate`** | Lag, Diff, Diff2, Momentum, ROC, LogReturn, Return | NaN stored in lookback; propagates through positional offset |
| **`nan-aware`** | FillNa, Ffill | NaN is the input these functions were designed to handle |

The library is strictly causal - `bfill` does not exist because backward fill
requires future data.

For dropping NaNs **across streams** see the `dropna` section at the end.

In [ ]:
import numpy as np
from screamer import RollingMean, Diff, FillNa, Ffill
from screamer import dropna

# Small deterministic test vector with one mid-stream NaN at index 3
x = np.array([1.0, 2.0, 3.0, np.nan, 5.0, 6.0])
print("input:", x)

## Policy 1: `ignore`

**Rule.** When a NaN arrives, the function emits NaN at that index and leaves
its internal state untouched (the sample is not stored in the lookback, not
added to running sums). Recovery is immediate: the very next finite input
produces a finite output.

Used by every summary-statistic function: all `Rolling*`, `Ew*`, `Cum*`,
bands, oscillators, and filters.

In [ ]:
out_ignore = RollingMean(3)(x)
print("RollingMean(3):", np.round(out_ignore, 4))
# Expected: [nan, nan, 2.0, nan, 3.3333, 4.6667]
#   indices 0-1 are warmup NaNs (window not yet full)
#   index 3 is NaN because input was NaN
#   index 4 recovers immediately - internal state was not corrupted

# Pin the ignore behaviour: exactly one mid-stream NaN at the gap index
assert np.isnan(out_ignore[3]), "gap index must be NaN"
assert not np.isnan(out_ignore[4]), "immediate recovery after gap"
assert not np.isnan(out_ignore[5]), "subsequent outputs are finite"
# total NaNs = 2 warmup + 1 gap = 3
assert np.sum(np.isnan(out_ignore)) == 3, "exactly 3 NaNs (2 warmup + 1 gap)"
print("ignore assertions passed")

## Policy 2: `propagate`

**Rule.** The function stores every input - including NaN - in its lookback
buffer. Output is computed by the positional formula (e.g. `x[t] - x[t-1]`)
with IEEE arithmetic propagating NaN naturally. One NaN in the input produces
NaN outputs for as long as that NaN remains inside the lookback window, then
full recovery.

Used by positional-offset functions: `Lag`, `Diff`, `Diff2`, `Momentum`,
`ROC`, `ROCP`, `ROCR`, `LogReturn`, `Return`.

**Why not `ignore`?** `Diff(1)` contractually means `x[t] - x[t-1]`. If the
NaN were silently dropped from the buffer, `Diff(1)` would compute a subtraction
across a gap - a wrong-but-finite answer. `propagate` gives a visible NaN instead.

In [ ]:
out_prop = Diff(1)(x)
print("Diff(1):", out_prop)
# Expected: [nan, 1.0, 1.0, nan, nan, 1.0]
#   index 0: NaN (warmup - no previous sample)
#   index 3: NaN (input is NaN → output is NaN)
#   index 4: NaN (previous sample was NaN → NaN still in lookback)
#   index 5: 1.0 (NaN has slid out of the lag-1 lookback → full recovery)

# Pin the propagate behaviour
assert np.isnan(out_prop[0]), "warmup NaN at index 0"
assert out_prop[1] == 1.0 and out_prop[2] == 1.0, "finite values before gap"
assert np.isnan(out_prop[3]), "NaN at the gap index"
assert np.isnan(out_prop[4]), "NaN propagates one step (lag-1 lookback)"
assert out_prop[5] == 1.0, "full recovery once NaN slides out of lookback"
# one input NaN → two extra NaN outputs (indices 3 and 4)
assert np.sum(np.isnan(out_prop)) == 3, "3 NaNs total (1 warmup + 2 from gap)"
print("propagate assertions passed")

## Policy 3: `nan-aware`

**Rule.** The function's purpose is to *consume* NaN and produce a finite
output. NaN input is what these functions were designed to handle; by design
they never pass NaN through.

Currently: `FillNa` (replace with a constant) and `Ffill` (forward-fill the
last finite value).

In [ ]:
out_fillna = FillNa(0.0)(x)   # replace NaN with 0
out_ffill  = Ffill()(x)       # carry last finite value forward

print("FillNa(0.0):", out_fillna)
print("Ffill()    :", out_ffill)

# Pin nan-aware behaviour: no NaN in output
assert not np.isnan(out_fillna).any(), "FillNa output must be NaN-free"
assert not np.isnan(out_ffill).any(),  "Ffill output must be NaN-free"

np.testing.assert_array_equal(out_fillna, [1.0, 2.0, 3.0, 0.0, 5.0, 6.0])
np.testing.assert_array_equal(out_ffill,  [1.0, 2.0, 3.0, 3.0, 5.0, 6.0])

# shape is preserved in both cases
assert out_fillna.shape == x.shape
assert out_ffill.shape  == x.shape
print("nan-aware assertions passed")

## Across streams: `dropna`

The three policies above operate *within* a single stream. The multi-stream
layer adds `dropna` for removing events whose value is NaN entirely, producing
a shorter but gap-free stream. This is a cardinality-changing operation:
output length < input length. The return is values-first: `(values, index)`.


In [ ]:
idx = np.arange(x.size, dtype=np.int64)
dv, dk = dropna(x, index=idx)

assert dv.size == x.size - 1, "one NaN removed -> one fewer event"
assert not np.isnan(dv).any(), "dropped stream is NaN-free"
print("dropped values:", dv)
print("dropped index :", dk)


**Takeaways**

- **`ignore`** (Rolling\*, Ew\*, Cum\*, bands, oscillators, filters): one NaN
  in → one NaN out; internal state is unchanged; recovery is immediate.
- **`propagate`** (Lag, Diff, Diff2, Momentum, ROC, LogReturn, Return): NaN is
  stored in the positional lookback and propagates via IEEE arithmetic; recovery
  once the NaN slides out of the window.
- **`nan-aware`** (FillNa, Ffill): NaN input produces a finite output by design.
- **`dropna`** is the cross-stream tool: cardinality-changing, not shape-preserving.
- `bfill` does not exist - backward fill requires future data and violates causality.